# Retail Demand Forecasting with Google TimesFM 2.5 (Favorita)

This notebook is a production-style, end-to-end tutorial for SKU-store forecasting, inventory recommendation, and warehouse allocation.

## 1. Business Problem and Decision Context

A retailer operates thousands of item-store combinations. Every day the operations team must decide:

- How many units to order for each item-store pair
- Which warehouse should ship replenishment
- Which stores are likely to stock out without action

We forecast demand for **7 / 14 / 30 days**, then convert forecasts into inventory and allocation decisions.

## 2. What This Notebook Covers

1. Download Favorita data directly from the web (Kaggle API)
2. Build full-scale item-store panel data
3. Forecast with TimesFM 2.5 (core + optional XReg covariate path)
4. Evaluate against baselines with leakage-safe backtesting
5. Convert forecasts to reorder recommendations
6. Allocate stock with heuristic and MILP optimization
7. Optionally generate Kaggle submission format

> Assumption: You have accepted the Kaggle competition rules for `favorita-grocery-sales-forecasting`.

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from __future__ import annotations

from datetime import timedelta
from pathlib import Path

import pandas as pd
import polars as pl

from scripts.retail_demand_timesfm_favorita import (
    PipelineConfig,
    build_inventory_policy,
    build_submission,
    build_test_promo_lookup,
    ensure_paths,
    evaluate_backtest,
    forecast_horizon,
    get_anchor_date,
    get_model_and_compile,
    heuristic_allocation,
    load_metadata,
    milp_allocation,
    prepare_series_panel,
    set_global_seeds,
    download_favorita_data,
)

## 3. Environment Setup

Use `uv` for dependency management.

In [2]:
# Run once in the project root if needed:
# !uv venv
# !uv sync --locked --group applied --group xreg
# !python -m ipykernel install --user --name timesfm-retail --display-name "Python (timesfm-retail)"

## 4. Configuration

Tune these defaults for your machine and runtime budget.

In [3]:
DATA_ROOT = PROJECT_ROOT / "data"
ARTIFACTS_ROOT = PROJECT_ROOT / "artifacts/retail_demand_timesfm"

cfg = PipelineConfig(
    data_root=DATA_ROOT,
    artifacts_root=ARTIFACTS_ROOT,
    horizons=(7, 14, 30),
    context_len=256,
    per_core_batch_size=8,
    forecast_batch_size=64,
    seed=42,
    device="auto",         # auto | cuda | cpu
    use_xreg=True,          # TimesFM covariate path
    xreg_mode="xreg + timesfm",
    xreg_ridge=1e-3,
    run_milp=True,
    include_submission=True,
)

paths = ensure_paths(cfg)
set_global_seeds(cfg.seed)
cfg

PipelineConfig(data_root=PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/data'), artifacts_root=PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm'), horizons=(7, 14, 30), context_len=256, per_core_batch_size=8, forecast_batch_size=64, seed=42, device='auto', use_xreg=True, xreg_mode='xreg + timesfm', xreg_ridge=0.001, run_milp=True, n_warehouses=5, warehouse_capacity_factor=1.1, service_level=0.95, lead_time_days=7, max_series=None, validation_series_limit=50000, include_submission=True)

## 5. Download Favorita Data from the Web

This cell downloads and extracts required competition files directly from Kaggle.

In [4]:
download_favorita_data(paths["raw"])
print("Downloaded files:")
print(sorted([p.name for p in paths["raw"].glob("*.csv")]))

2026-07-03 00:15:20.003 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found train.csv, skipping download.


2026-07-03 00:15:20.003 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found test.csv, skipping download.


2026-07-03 00:15:20.004 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found items.csv, skipping download.


2026-07-03 00:15:20.004 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found stores.csv, skipping download.


2026-07-03 00:15:20.004 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found oil.csv, skipping download.


2026-07-03 00:15:20.005 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found holidays_events.csv, skipping download.


2026-07-03 00:15:20.005 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found transactions.csv, skipping download.


2026-07-03 00:15:20.005 | INFO     | scripts.retail_demand_timesfm_favorita:download_favorita_data:213 - Found sample_submission.csv, skipping download.


Downloaded files:
['holidays_events.csv', 'items.csv', 'oil.csv', 'sample_submission.csv', 'stores.csv', 'test.csv', 'train.csv', 'transactions.csv']


## 6. Build Full-Scale Panel Dataset

We filter training rows to item-store pairs present in test and materialize:

- `train_long_filtered.parquet`
- `series_panel.parquet` (one row per series, list columns)

In [5]:
train_long_path, panel_path = prepare_series_panel(
    raw_dir=paths["raw"],
    processed_dir=paths["processed"],
    max_series=cfg.max_series,
)

anchor = get_anchor_date(train_long_path)
print("Train long:", train_long_path)
print("Series panel:", panel_path)
print("Forecast anchor date:", anchor)

2026-07-03 00:15:20.009 | INFO     | scripts.retail_demand_timesfm_favorita:prepare_series_panel:270 - Found /home/ahmad/AI/Github/google-TimesFM-implementation/data/processed/train_long_filtered.parquet


2026-07-03 00:15:20.009 | INFO     | scripts.retail_demand_timesfm_favorita:prepare_series_panel:286 - Found /home/ahmad/AI/Github/google-TimesFM-implementation/data/processed/series_panel.parquet


Train long: /home/ahmad/AI/Github/google-TimesFM-implementation/data/processed/train_long_filtered.parquet
Series panel: /home/ahmad/AI/Github/google-TimesFM-implementation/data/processed/series_panel.parquet
Forecast anchor date: 2017-08-20


## 7. Metadata and Feature Inputs

Load item/store metadata and test promotions used by XReg covariate forecasting.

In [6]:
items_df, stores_df, test_df = load_metadata(paths["raw"])
family_by_item = dict(zip(items_df["item_nbr"].astype(int), items_df["family"].astype(str)))
cluster_by_store = dict(zip(stores_df["store_nbr"].astype(int), stores_df["cluster"].astype(int)))
perishable_weight_by_item = dict(
    zip(
        items_df["item_nbr"].astype(int),
        (items_df["perishable"].astype(int).map({1: 1.25, 0: 1.0})).astype(float),
    )
)
test_promo_lookup = build_test_promo_lookup(test_df)

items_df.head(), stores_df.head(), test_df.head()

(   item_nbr     family  class  perishable
 0      1001  GROCERY I   1000           0
 1      1002  BEVERAGES   1001           1,
    store_nbr       city      state type  cluster
 0          1      Quito  Pichincha    A        1
 1          2      Quito  Pichincha    B        2
 2          3  Guayaquil     Guayas    C        3,
           id       date  store_nbr  item_nbr  onpromotion
 0  125497040 2017-08-21          1      1001        False
 1  125497041 2017-08-21          1      1002        False
 2  125497042 2017-08-21          2      1001        False
 3  125497043 2017-08-21          2      1002        False
 4  125497044 2017-08-21          3      1001        False)

## 8. Load and Compile TimesFM 2.5

We compile once at `max_horizon = max(30, 16)` because we also build optional 16-day submission outputs.

In [7]:
submission_horizon = 16 if cfg.include_submission else 0
max_horizon = max(max(cfg.horizons), submission_horizon)

model = get_model_and_compile(
    cfg=cfg,
    max_horizon=max_horizon,
    enable_backcast=cfg.use_xreg,
)

2026-07-03 00:15:23.351 | INFO     | scripts.retail_demand_timesfm_favorita:get_model_and_compile:367 - TimesFM compiled with context=256 max_horizon=30 xreg=True


## 9. Forecast 7 / 14 / 30 Days

This is the main production inference stage.

In [8]:
forecast_paths = {}
stats_paths = {}

for horizon in cfg.horizons:
    forecast_path, stats_path = forecast_horizon(
        model=model,
        panel_path=panel_path,
        output_dir=paths["artifacts"],
        anchor_date=anchor,
        horizon=horizon,
        cfg=cfg,
        family_by_item=family_by_item,
        cluster_by_store=cluster_by_store,
        test_promo_lookup=test_promo_lookup,
    )
    forecast_paths[horizon] = forecast_path
    stats_paths[horizon] = stats_path

forecast_paths

H7: 0it [00:00, ?it/s]

2026-07-03 00:15:26.100 | INFO     | scripts.retail_demand_timesfm_favorita:forecast_horizon:616 - Wrote horizon 7 forecast: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_7/forecast_h7.parquet


H14: 0it [00:00, ?it/s]

2026-07-03 00:15:26.330 | INFO     | scripts.retail_demand_timesfm_favorita:forecast_horizon:616 - Wrote horizon 14 forecast: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/forecast_h14.parquet


H30: 0it [00:00, ?it/s]

2026-07-03 00:15:26.552 | INFO     | scripts.retail_demand_timesfm_favorita:forecast_horizon:616 - Wrote horizon 30 forecast: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_30/forecast_h30.parquet


{7: PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_7/forecast_h7.parquet'),
 14: PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/forecast_h14.parquet'),
 30: PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_30/forecast_h30.parquet')}

## 10. Validation vs Baselines (Leakage-Safe)

We run historical backtest and compare TimesFM against:

- Last-value naive
- Seasonal-7 naive

Metrics include weighted RMSLE-style score and WMAPE.

In [9]:
metrics_path = evaluate_backtest(
    model=model,
    panel_path=panel_path,
    train_long_path=train_long_path,
    artifacts_dir=paths["artifacts"],
    cfg=cfg,
    family_by_item=family_by_item,
    cluster_by_store=cluster_by_store,
    perishable_weight_by_item=perishable_weight_by_item,
)

pd.read_csv(metrics_path)

Backtest: 0it [00:00, ?it/s]

2026-07-03 00:15:26.638 | INFO     | scripts.retail_demand_timesfm_favorita:evaluate_backtest:770 - Backtest metrics saved: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/validation_metrics.csv


,model,horizon,nwrmsle,wmape,rows
0,naive_seasonal7,7,0.128321,0.114922,42
1,naive_last,7,0.167761,0.140153,42
2,tfm,7,2.789506,1.000000,42
3,naive_seasonal7,14,0.139561,0.118784,84
4,naive_last,14,0.184042,0.149969,84
5,tfm,14,2.798948,1.000000,84
6,naive_seasonal7,30,0.143838,0.117535,180
7,naive_last,30,0.186610,0.153444,180
8,tfm,30,2.788600,1.000000,180


## 11. Inventory Policy from Forecast Uncertainty

For each item-store, we compute:

- Lead-time demand estimate
- Safety stock (quantile spread aware)
- Reorder point
- Recommended order quantity

In [10]:
h7 = min(cfg.horizons)
inventory_path = build_inventory_policy(
    forecast_h7_path=forecast_paths[h7],
    series_stats_path=stats_paths[h7],
    stores_df=stores_df,
    items_df=items_df,
    cfg=cfg,
    artifacts_dir=paths["artifacts"],
)

inv_df = pl.read_parquet(inventory_path).to_pandas()
inv_df.head()

2026-07-03 00:15:26.664 | INFO     | scripts.retail_demand_timesfm_favorita:build_inventory_policy:832 - Inventory policy saved: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/inventory_policy.parquet


,store_nbr,item_nbr,lead_point_demand,lead_q50_demand,lead_q90_demand,week_point_demand,trailing_mean_14,context_sum_7,safety_stock,estimated_on_hand,reorder_point,recommended_order_qty,warehouse_id,family,priority_weight
0,1,1001,80.905464,80.905464,119.006058,80.905464,13.386428,95.440002,38.100594,56.222997,143.390442,87.167445,WH_01,GROCERY I,1.00
1,1,1002,84.868484,84.868484,119.681595,84.868484,14.793571,102.130005,34.813110,62.132996,141.961990,79.828994,WH_01,BEVERAGES,1.25
2,2,1001,90.170082,90.170082,130.939056,90.170082,14.709285,106.599998,40.768974,61.778996,157.031189,95.252193,WH_02,GROCERY I,1.00
3,2,1002,96.704124,96.704124,141.816101,96.704124,16.177143,115.440002,45.111977,67.944001,170.687775,102.743774,WH_02,BEVERAGES,1.25
4,3,1001,92.309540,92.309540,128.922394,92.309540,15.205001,109.309998,36.612854,63.861004,152.354614,88.493611,WH_03,GROCERY I,1.00


## 12. Warehouse Allocation: Heuristic and MILP

- **Heuristic**: fast rule-based fill from primary warehouse then spillover
- **MILP**: constrained optimization for allocation cost + shortage penalty

In [11]:
heuristic_path = heuristic_allocation(
    inventory_path=inventory_path,
    stores_df=stores_df,
    cfg=cfg,
    artifacts_dir=paths["artifacts"],
)

milp_path = None
if cfg.run_milp:
    milp_path = milp_allocation(
        inventory_path=inventory_path,
        stores_df=stores_df,
        cfg=cfg,
        artifacts_dir=paths["artifacts"],
    )

print("Heuristic allocation:", heuristic_path)
print("MILP allocation:", milp_path)

2026-07-03 00:15:26.684 | INFO     | scripts.retail_demand_timesfm_favorita:heuristic_allocation:917 - Heuristic allocation saved: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/warehouse_allocation_heuristic.parquet


2026-07-03 00:15:26.696 | INFO     | scripts.retail_demand_timesfm_favorita:milp_allocation:1011 - MILP allocation saved: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/warehouse_allocation_milp.parquet


Heuristic allocation: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/warehouse_allocation_heuristic.parquet
MILP allocation: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/warehouse_allocation_milp.parquet


## 13. Optional Kaggle Submission (16-Day Horizon)

Generates `id,unit_sales` CSV using TimesFM forecast mapped to test horizon days.

In [12]:
submission_path = None
if cfg.include_submission:
    f16_path, _ = forecast_horizon(
        model=model,
        panel_path=panel_path,
        output_dir=paths["artifacts"],
        anchor_date=anchor,
        horizon=16,
        cfg=cfg,
        family_by_item=family_by_item,
        cluster_by_store=cluster_by_store,
        test_promo_lookup=test_promo_lookup,
    )
    submission_path = build_submission(
        forecast_h16_path=f16_path,
        raw_dir=paths["raw"],
        anchor_date=anchor,
        artifacts_dir=paths["artifacts"],
    )

submission_path

H16: 0it [00:00, ?it/s]

2026-07-03 00:15:26.848 | INFO     | scripts.retail_demand_timesfm_favorita:forecast_horizon:616 - Wrote horizon 16 forecast: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_16/forecast_h16.parquet


2026-07-03 00:15:26.856 | INFO     | scripts.retail_demand_timesfm_favorita:build_submission:1040 - Submission saved: /home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/submission_timesfm.csv


PosixPath('/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/submission_timesfm.csv')

## 14. Inspect Outputs

All artifacts are under `artifacts/retail_demand_timesfm/`.

In [13]:
for p in sorted(paths["artifacts"].rglob("*")):
    if p.is_file():
        print(p)

/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/forecast_h14.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/parts/forecast_part_000000.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/series_stats.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_14/stats_parts/series_stats_000000.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_16/forecast_h16.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_16/parts/forecast_part_000000.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_16/series_stats.parquet
/home/ahmad/AI/Github/google-TimesFM-implementation/artifacts/retail_demand_timesfm/horizon_16/stats_parts/series_stats_000000.parquet
/home/

## 15. Next Operationalization Steps

1. Schedule this pipeline daily and persist forecasts in your planning DB.
2. Replace proxy warehouse mapping with real network topology and transport costs.
3. Add service-level policies by product family and business criticality.
4. Track model drift and backtest metrics over time.